In [425]:
import numpy as np
import pandas as pd

# Get the absolute directory of the current file
df = pd.read_csv("../datasets/kickstarter_projects.csv")
df.columns = df.columns.str.strip()


df_transformed = df.copy()

#Transforms string datetime pandas datetime object
df_transformed[["Launched", "Deadline"]] = df_transformed[["Launched", "Deadline"]].apply(pd.to_datetime)
numeric_values = df_transformed[["Goal", "Pledged", "Backers"]].to_numpy() #Retrieves only known numeric columns


df_transformed[["Launched", "Deadline", "Goal", "Pledged", "Backers"]].head(10)


,Launched,Deadline,Goal,Pledged,Backers
0,2009-04-21 21:02:48,2009-05-31,1000,625,30
1,2009-04-23 00:07:53,2009-07-20,80000,22,3
2,2009-04-24 21:52:03,2009-05-03,20,35,3
3,2009-04-25 17:36:21,2009-07-14,99,145,25
4,2009-04-27 14:10:39,2009-05-26,1900,387,10
5,2009-04-28 13:55:41,2009-05-16,3000,3329,110
6,2009-04-29 02:04:21,2009-05-29,200,41,3
7,2009-04-29 02:58:50,2009-05-29,500,563,18
8,2009-04-29 04:37:37,2009-06-01,500,0,0
9,2009-04-29 05:26:32,2009-05-16,300,15,2


### Task 1. Inspect the Numerical Array

In [426]:
numerical_labels = ["Goal", "Pledged", "Backers"]


def report_numeric(numeric_val, label):
    print(f"Report for {label}")
    return pd.DataFrame({
        "Property": [
            "Number of observations",
            "Shape",
            "Size",
            "Data type",
            "Minimum",
            "Maximum",
            "Mean",
            "Median",
            "Standard deviation"
        ],
        "Suggested NumPy approach": [
            "len(values)",
            "values.shape",
            "values.size",
            "values.dtype",
            "np.min(values)",
            "np.max(values)",
            "np.mean(values)",
            "np.median(values)",
            "np.std(values)"
        ],
        "Result": [
            len(numeric_val),
            numeric_val.shape,
            numeric_val.size,
            numeric_val.dtype,
            np.min(numeric_val),
            np.max(numeric_val),
            np.mean(numeric_val),
            np.median(numeric_val),
            np.std(numeric_val)
        ]
    })



[print(f"{report_numeric(numeric_val, label)}\n\n") for numeric_val, label in zip(numeric_values.T, numerical_labels)] #Reports every numerical column

Report for Goal
                 Property Suggested NumPy approach          Result
0  Number of observations              len(values)          374853
1                   Shape             values.shape       (374853,)
2                    Size              values.size          374853
3               Data type             values.dtype           int64
4                 Minimum           np.min(values)               0
5                 Maximum           np.max(values)       166361391
6                    Mean          np.mean(values)    45863.784796
7                  Median        np.median(values)          5500.0
8      Standard deviation           np.std(values)  1158776.633097


Report for Pledged
                 Property Suggested NumPy approach        Result
0  Number of observations              len(values)        374853
1                   Shape             values.shape     (374853,)
2                    Size              values.size        374853
3               Data type        

[None, None, None]

### Task 2. Vectorized Computation

#### Domain-Specific Calculation

In [427]:
df_transformed["days_before_deadline"] = (df_transformed["Deadline"] - df_transformed["Launched"]).dt.days
print(f"Shape of dataframe before filtering of days before deadline:{df_transformed.shape}")
df_transformed[df_transformed["days_before_deadline"] > 0].head() #removes some of the row?
print(f"Shape of dataframe after filtering of days before deadline:{df_transformed.shape}")

Shape of dataframe before filtering of days before deadline:(374853, 12)
Shape of dataframe after filtering of days before deadline:(374853, 12)


#### Unit Conversion
* I calculated the distribution of each state to the whole dataset in percentage and verified that the summation of percentage across states is equivalent to 100%

In [428]:
status_df = pd.DataFrame({
    'Count': df_transformed['State'].value_counts(),
    'Percentage': (df_transformed['State'].value_counts(normalize=True) * 100).round(0)
})

status_df.loc['Total'] = [
    status_df['Count'].sum(),
    status_df['Percentage'].sum().astype(str) + "%"
]
status_df

,Count,Percentage
State,,
Failed,197611,53.0
Successful,133851,36.0
Canceled,38751,10.0
Live,2798,1.0
Suspended,1842,0.0
Total,374853,100.0%


# Bonus: Anomaly Detection
## What Was Found Out
* I calculated first the ratio of pledged and goal (percent_to_goal). With this I filtered the records that their state column is NOT Successful
* Then from that filtered copy of the dataframe, I then filtered records that are not succesful but reaches their goal (The ratio in percent is >= 100%)
* I also checked the records where they have achieved their goal and is within the deadline (Produces 2 records only)
* I also checked the records that are unsuccessful and haven't reached their deadline yet (Produces 1437 records)
* The record that makes sense to me are the ones that havent achieved their goal (percent to goal < 100 )

In [429]:
print(f"Shape of dataframe before removing successful:{df_transformed.shape}")
df_transformed_not_success = df_transformed[df_transformed['State'] != 'Successful'].copy() #Removes data that are successful (Pertains only rows that are successful to check for anomaly)
print(f"Shape of dataframe after removing successful:{df_transformed_not_success.shape}")

df_transformed_not_success['percent_to_goal'] = ((df_transformed_not_success['Pledged'] / df_transformed_not_success['Goal']) * 100).round(2) #Calculates the ratio of pledge to its goal for unsuccessful records to determine how close it is to achieving its goal

print(f"{df_transformed_not_success.shape}")



failed_but_achieved_goal_df_reaches_deadline = df_transformed_not_success[(df_transformed_not_success["percent_to_goal"] >= 100) & (df_transformed_not_success["days_before_deadline"] == 0)]
print(f"Number of records that achieve their goals but reaches deadline but are labeled as unsucessful: {failed_but_achieved_goal_df_reaches_deadline.shape}")

failed_but_achieved_goal_df = df_transformed_not_success[(df_transformed_not_success["percent_to_goal"] >= 100) & (df_transformed_not_success["days_before_deadline"] > 0)]
print(f"Number of records that achieve their goals but are labeled as unsucessful: {failed_but_achieved_goal_df.shape}")

print(failed_but_achieved_goal_df[failed_but_achieved_goal_df["State"] == "Failed"][["Goal", "Pledged", "days_before_deadline", "State"]])


truly_unsucessful = df_transformed_not_success[(df_transformed_not_success["percent_to_goal"] < 100) & (df_transformed_not_success["days_before_deadline"] >= 0)]
print(f"Number of records that is truly not successful: {truly_unsucessful.shape}")


Shape of dataframe before removing successful:(374853, 12)
Shape of dataframe after removing successful:(241002, 12)
(241002, 13)
Number of records that achieve their goals but reaches deadline but are labeled as unsucessful: (2, 13)
Number of records that achieve their goals but are labeled as unsucessful: (1437, 13)
         Goal  Pledged  days_before_deadline   State
2295    10000    10797                    58  Failed
5754     3600     3600                    29  Failed
25983   40000    40000                    59  Failed
33823    3000     3088                    57  Failed
62467    3000     3000                    21  Failed
86441   23000    23020                    29  Failed
108392   6000     6183                    40  Failed
151769     10       10                    29  Failed
259754  12000    12000                    49  Failed
Number of records that is truly not successful: (239561, 13)


In [430]:
np_not_success = truly_unsucessful[["Pledged", "Goal", "percent_to_goal"]].to_numpy()
print(f"Shape of np_not_success before transposing: {np_not_success.shape}")

np_not_success = np_not_success.T
print(f"Shape of np_not_success after transposing: {np_not_success.shape}") #The array is still a 2d array but each row is now the whole column record

Shape of np_not_success before transposing: (239561, 3)
Shape of np_not_success after transposing: (3, 239561)


# Task 3 BOOLEAN MASKING
## THE GOAL: Retrieve The Top 25 in terms of percentile out of all not successful records 

In [431]:
p75 = np.percentile(np_not_success[2, :], 75)
mask = np_not_success[2, :] > p75 #Boolean masking definition (This retrieves the top 25 percent based on row 2 which is the percent_to_goal)
above_75th_percentile = np_not_success[:,mask]

print(f"Shape of unfiltered unsuccessful records: {np_not_success.shape}")
print(f"75th percentile: {p75}")
print(f"Shape of above_75th_percentile: {above_75th_percentile.shape}\n\n") #There are 59915 records at this percentile

print(f"Mean of above 75th percentile: {np.mean(above_75th_percentile[2, :])}")
print(f"Standard deviation of above 75th percentile: {np.std(above_75th_percentile[2, :])}")


Shape of unfiltered unsuccessful records: (3, 239561)
75th percentile: 10.84
Shape of above_75th_percentile: (3, 59887)


Mean of above 75th percentile: 30.178364252675873
Standard deviation of above 75th percentile: 17.750532114687083


In [432]:
import time
top_75_list = list(np_not_success[2, :])

top_75 = []

start = time.perf_counter()
top_75_list = [i for i in top_75_list if i >= p75]
end = time.perf_counter()
print(f"Seconds it took in order to calculate using loop: {end - start}")


start = time.perf_counter()
top_25 =  np_not_success[:,mask] #Boolean masking using numpy (This are the top 25 for the whole data that)
end = time.perf_counter()
print(f"Seconds it took in order to calculate vector operation using numpy: {end - start}")


Seconds it took in order to calculate using loop: 0.01092866700491868
Seconds it took in order to calculate vector operation using numpy: 0.0012744999839924276
